In [13]:
import pandas as pd
import numpy as np

df = pd.read_csv("cleaned_helpdesk_tickets.csv")

In [2]:
df.columns

Index(['ticket_id', 'date_created', 'date_resolved', 'category', 'subcategory',
       'priority', 'status', 'assigned_team', 'resolution_time_hrs',
       'description', 'escalated', 'complexity'],
      dtype='object')

In [3]:
df.head(10)

,ticket_id,date_created,date_resolved,category,subcategory,priority,status,assigned_team,resolution_time_hrs,description,escalated,complexity
0,1,2023-11-16,2023-11-23,Access,Access denied by auditor policy,Low,Open,Desktop Support,201.60,User reported Access denied by auditor policy....,True,High
1,2,2023-11-29,2023-12-02,Security,Suspicious DLL injection,Critical,Open,Security Team,75.60,User reported Suspicious DLL injection. Furthe...,False,Low
2,3,2023-11-29,2023-12-06,Security,Suspicious SOC alert,Low,Pending,Security Team,176.40,User reported Suspicious SOC alert. Further in...,True,Low
3,4,2023-11-04,2023-11-09,Software,App freezing,Low,On Hold,Application Support,84.00,User reported App freezing. Further investigat...,False,Low
4,5,2023-11-03,2023-11-10,Access,Access logging failure,Critical,In Progress,Desktop Support,134.40,User reported Access logging failure. Further ...,True,Medium
5,6,2023-11-04,NaN,Security,Suspicious outbound traffic,Critical,Pending,Security Team,NaN,User reported Suspicious outbound traffic. Fur...,False,Low
6,7,2023-11-20,2023-11-25,Security,Unauthorized access attempt,Medium,Open,Security Team,180.00,User reported Unauthorized access attempt. Fur...,True,Medium
7,8,2023-11-17,2023-11-20,Hardware,Peripheral device failure,High,In Progress,Desktop Support,57.60,User reported Peripheral device failure. Furth...,False,Medium
8,9,2023-11-28,2023-12-05,Network,Access point down,High,Open,Network Team,152.88,User reported Access point down. Further inves...,True,Low
9,10,2023-11-30,2023-12-05,Security,Suspicious session hijack,Medium,Pending,Security Team,270.00,User reported Suspicious session hijack. Furth...,True,High


In [4]:
df[['assigned_team', 'status', 'priority', 'resolution_time_hrs']].head(10)

,assigned_team,status,priority,resolution_time_hrs
0,Desktop Support,Open,Low,201.60
1,Security Team,Open,Critical,75.60
2,Security Team,Pending,Low,176.40
3,Application Support,On Hold,Low,84.00
4,Desktop Support,In Progress,Critical,134.40
5,Security Team,Pending,Critical,NaN
6,Security Team,Open,Medium,180.00
7,Desktop Support,In Progress,High,57.60
8,Network Team,Open,High,152.88
9,Security Team,Pending,Medium,270.00


In [5]:
df.head(10)

,ticket_id,date_created,date_resolved,category,subcategory,priority,status,assigned_team,resolution_time_hrs,description,escalated,complexity
0,1,2023-11-16,2023-11-23,Access,Access denied by auditor policy,Low,Open,Desktop Support,201.60,User reported Access denied by auditor policy....,True,High
1,2,2023-11-29,2023-12-02,Security,Suspicious DLL injection,Critical,Open,Security Team,75.60,User reported Suspicious DLL injection. Furthe...,False,Low
2,3,2023-11-29,2023-12-06,Security,Suspicious SOC alert,Low,Pending,Security Team,176.40,User reported Suspicious SOC alert. Further in...,True,Low
3,4,2023-11-04,2023-11-09,Software,App freezing,Low,On Hold,Application Support,84.00,User reported App freezing. Further investigat...,False,Low
4,5,2023-11-03,2023-11-10,Access,Access logging failure,Critical,In Progress,Desktop Support,134.40,User reported Access logging failure. Further ...,True,Medium
5,6,2023-11-04,NaN,Security,Suspicious outbound traffic,Critical,Pending,Security Team,NaN,User reported Suspicious outbound traffic. Fur...,False,Low
6,7,2023-11-20,2023-11-25,Security,Unauthorized access attempt,Medium,Open,Security Team,180.00,User reported Unauthorized access attempt. Fur...,True,Medium
7,8,2023-11-17,2023-11-20,Hardware,Peripheral device failure,High,In Progress,Desktop Support,57.60,User reported Peripheral device failure. Furth...,False,Medium
8,9,2023-11-28,2023-12-05,Network,Access point down,High,Open,Network Team,152.88,User reported Access point down. Further inves...,True,Low
9,10,2023-11-30,2023-12-05,Security,Suspicious session hijack,Medium,Pending,Security Team,270.00,User reported Suspicious session hijack. Furth...,True,High


# fixing NaN values in resolution time column 

In [6]:
df['resolution_time_hrs'] = df['resolution_time_hrs'].fillna(df['resolution_time_hrs'].median())

# assigning agents

In [7]:
import numpy as np

def assign_agent(team):
    if team == "Desktop Support":
        return np.random.choice(["D1","D2","D3","D4","D5"], p=[0.4,0.25,0.2,0.1,0.05])
    elif team == "Application Support":
        return np.random.choice(["A1","A2","A3","A4","A5"], p=[0.35,0.25,0.2,0.1,0.1])
    elif team == "Network Team":
        return np.random.choice(["N1","N2","N3","N4","N5"], p=[0.3,0.25,0.2,0.15,0.1])
    elif team == "Security Team":
        return np.random.choice(["S1","S2","S3","S4","S5"], p=[0.3,0.25,0.2,0.15,0.1])

df['agent_id'] = df['assigned_team'].apply(assign_agent)

In [8]:
df['agent_id'].value_counts().head(10)

agent_id
D1    160032
D2    100203
D3     79704
A1     70140
S1     59971
N1     59825
A2     50587
S2     50292
N2     49981
N3     39961
Name: count, dtype: int64

# building workload metrics

In [9]:
tickets_per_agent = df.groupby('agent_id').size().rename('ticket_count')

In [10]:
df['is_backlog'] = df['status'] != 'Resolved'

backlog_per_agent = df.groupby('agent_id')['is_backlog'].sum()

In [11]:
avg_resolution = df.groupby('agent_id')['resolution_time_hrs'].mean()

In [12]:
df['is_high_priority'] = df['priority'].isin(['High', 'Critical'])

priority_ratio = df.groupby('agent_id')['is_high_priority'].mean()

In [14]:
metrics_df = pd.concat([
    tickets_per_agent,
    backlog_per_agent,
    avg_resolution,
    priority_ratio
], axis=1)

metrics_df.columns = [
    'ticket_count',
    'backlog_count',
    'avg_resolution_time',
    'high_priority_ratio'
]

metrics_df = metrics_df.reset_index()

In [17]:
metrics_df.head()

,agent_id,ticket_count,backlog_count,avg_resolution_time,high_priority_ratio
0,A1,70140,55969,88.760719,0.498004
1,A2,50587,40486,88.516006,0.499298
2,A3,39801,31730,88.981262,0.498832
3,A4,20148,16146,88.583085,0.497618
4,A5,19952,15937,89.319286,0.493785


In [18]:
df['status'].value_counts()

status
Resolved       200903
On Hold        200519
Open           199802
In Progress    199435
Pending        199341
Name: count, dtype: int64

# rule based classification

In [19]:
ticket_threshold_high = metrics_df['ticket_count'].quantile(0.75)
ticket_threshold_low = metrics_df['ticket_count'].quantile(0.25)

backlog_threshold = metrics_df['backlog_count'].quantile(0.75)

In [20]:
def classify_workload(row):
    if row['ticket_count'] > ticket_threshold_high or row['backlog_count'] > backlog_threshold:
        return 'Overloaded'
    elif row['ticket_count'] < ticket_threshold_low:
        return 'Underloaded'
    else:
        return 'Balanced'

In [21]:
metrics_df['workload_status'] = metrics_df.apply(classify_workload, axis=1)

In [43]:
metrics_df['workload_status'].value_counts()

workload_status
Balanced       10
Overloaded      5
Underloaded     5
Name: count, dtype: int64

In [24]:
metrics_df.sort_values(by='ticket_count', ascending=False)

,agent_id,ticket_count,backlog_count,avg_resolution_time,high_priority_ratio,workload_status
5,D1,160032,128069,76.315849,0.500144,Overloaded
6,D2,100203,80073,76.557660,0.499247,Overloaded
7,D3,79704,63659,76.195760,0.497742,Overloaded
0,A1,70140,55969,88.760719,0.498004,Overloaded
15,S1,59971,48066,121.186233,0.496223,Overloaded
10,N1,59825,47701,107.817484,0.499056,Balanced
1,A2,50587,40486,88.516006,0.499298,Balanced
16,S2,50292,40122,120.196516,0.500795,Balanced
11,N2,49981,39893,108.533886,0.502391,Balanced
12,N3,39961,31838,107.969724,0.500338,Balanced


# Decision tree model (Prepare features-X and Target-Y)

In [25]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = metrics_df[['ticket_count', 'backlog_count', 'avg_resolution_time', 'high_priority_ratio']]
y = metrics_df['workload_status']

# encoding target label

In [26]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# train test split - introducing a stratified split

In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.3,
    random_state=42,
    stratify=y_encoded
)

# Training Decision Tree

In [28]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current

# Predictions

In [39]:
y_pred = model.predict(X_test)

#confusion matrix

In [44]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[3 0 0]
 [1 1 0]
 [0 0 1]]


In [40]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print(classification_report(y_test, y_pred))

Accuracy: 0.8333333333333334
              precision    recall  f1-score   support

           0       0.75      1.00      0.86         3
           1       1.00      0.50      0.67         2
           2       1.00      1.00      1.00         1

    accuracy                           0.83         6
   macro avg       0.92      0.83      0.84         6
weighted avg       0.88      0.83      0.82         6



In [41]:
accuracy

0.8333333333333334

In [42]:
print(classification_report(
    y_test,
    y_pred,
    labels=[0,1,2],
    target_names=le.classes_
))

              precision    recall  f1-score   support

    Balanced       0.75      1.00      0.86         3
  Overloaded       1.00      0.50      0.67         2
 Underloaded       1.00      1.00      1.00         1

    accuracy                           0.83         6
   macro avg       0.92      0.83      0.84         6
weighted avg       0.88      0.83      0.82         6

